# Lesson 02: The maths behind AI

*Worked solutions for [the lesson page](../../lessons/02-math-for-ai.html). The exercise version without solutions is `../02-math-for-ai.ipynb`.*

Just enough linear algebra, calculus and probability to understand what every model is doing, with each idea turned straight into NumPy code.

**Runs on:** Any computer (CPU).

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy

## Vectors and similarity

### Length and angle

That ratio is **cosine similarity**: 1 when the vectors point the same way, 0 when they are perpendicular (unrelated), −1 when opposite. It ignores length and compares only direction, which is exactly what you want when comparing the meaning of two texts of different lengths.

In [ ]:
import numpy as np

a = np.array([3.0, 4.0])
b = np.array([4.0, 3.0])
print(np.linalg.norm(a))                 # 5.0
print(a @ b)                             # 24.0
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(cos)                               # 0.96: nearly the same direction

> **Normalise once, then dot**
>
> If you scale every vector to length 1 up front, cosine similarity becomes a plain dot product. Vector databases rely on this trick: store normalised embeddings, and similarity search becomes one fast matrix multiplication.

## Matrices as transformations

A matrix is a grid of numbers, but the useful way to think of one is as a *function that transforms vectors*. Multiplying a vector by a matrix rotates, stretches, squashes or projects it.

In [ ]:
R = np.array([[0, -1],
              [1,  0]])                  # rotate 90 degrees anticlockwise
print(R @ np.array([1, 0]))              # [0 1]

S = np.array([[2, 0],
              [0, 0.5]])                 # stretch x, squash y
print(S @ np.array([1, 1]))              # [2.  0.5]

### Rank and the SVD, in one paragraph

Any matrix can be factored as $A = U\Sigma V^\top$, the **singular value decomposition**. $U$ and $V$ are rotations and $\Sigma$ is a diagonal of non-negative *singular values* that say how much the matrix stretches along each direction. The number of non-zero singular values is the **rank**: how many independent directions the matrix really uses. A huge matrix with only a few large singular values is "secretly small", and you can approximate it with far fewer numbers. That single idea underlies PCA (lesson 04) and LoRA fine-tuning (lesson 18).

In [ ]:
rng = np.random.default_rng(0)
A = rng.normal(size=(100, 20)) @ rng.normal(size=(20, 50))   # a 100x50 matrix built from rank 20
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print((S > 1e-8).sum())          # 20: only 20 directions are really used

## Calculus: how models learn

### Gradient descent in one dimension

where $\eta$ (eta) is the **learning rate**, the step size.

In [ ]:
def f(w):  return (w - 3) ** 2 + 1      # minimum at w = 3
def df(w): return 2 * (w - 3)            # its derivative

w, lr = -4.0, 0.1
for step in range(50):
    w -= lr * df(w)
print(round(w, 4))                       # 2.9999

### Worked example: fit a line with gradient descent

$$
L = \frac{1}{N}\sum_i (\hat{y}_i - y_i)^2,
\qquad
\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i (\hat{y}_i - y_i)\,x_i,
\qquad
\frac{\partial L}{\partial b} = \frac{2}{N}\sum_i (\hat{y}_i - y_i)
$$

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 200)
y = 2.5 * x + 4 + rng.normal(0, 2, 200)      # true line plus noise

w, b, lr = 0.0, 0.0, 0.01
for epoch in range(2000):
    err = (w * x + b) - y                     # prediction error, shape (200,)
    loss = (err ** 2).mean()
    dw = 2 * (err * x).mean()                 # dL/dw
    db = 2 * err.mean()                       # dL/db
    w -= lr * dw
    b -= lr * db
    if epoch % 500 == 0:
        print(epoch, round(loss, 3), round(w, 3), round(b, 3))

print(f"w={w:.3f}, b={b:.3f}")               # close to 2.45 and 4.09
print(np.polyfit(x, y, 1))                   # the exact least-squares answer agrees

### Hands-on: check a gradient numerically

Analytic gradients are easy to get wrong. A **gradient check** compares them with a numerical estimate using a tiny nudge $\varepsilon$:
$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta + \varepsilon e_i) - L(\theta - \varepsilon e_i)}{2\varepsilon}$.
Write `numerical_grad(loss_fn, params)` and confirm it matches `dw` and `db` above at `params = [1.0, 1.0]`.

#### Solution: a solution

In [ ]:
def numerical_grad(loss_fn, params, eps=1e-5):
    g = np.zeros_like(params)
    for i in range(len(params)):
        e = np.zeros_like(params)
        e[i] = eps
        g[i] = (loss_fn(params + e) - loss_fn(params - e)) / (2 * eps)
    return g

def loss_fn(p):
    w, b = p
    return ((w * x + b - y) ** 2).mean()

p = np.array([1.0, 1.0])
err = p[0] * x + p[1] - y
print(numerical_grad(loss_fn, p))
print([2 * (err * x).mean(), 2 * err.mean()])   # should agree to ~6 decimal places

Researchers still use gradient checks when they write custom layers. It is the unit test of calculus.

## Probability: what models output

### Bayes' rule

Classic example. A fault affects 1% of instruments. A diagnostic test catches 95% of faulty ones but also flags 5% of healthy ones. An instrument tests positive: how likely is it to be faulty?

In [ ]:
prior, sensitivity, false_pos = 0.01, 0.95, 0.05
posterior = sensitivity * prior / (sensitivity * prior + false_pos * (1 - prior))
print(round(posterior, 3))       # 0.161, only 16%

### Softmax: turning scores into probabilities

$$
\mathrm{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)    # subtract the max for numerical stability
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

print(softmax(np.array([2.0, 1.0, 0.1])).round(3))          # [0.659 0.242 0.099]
print(softmax(np.array([1000.0, 1001.0, 1002.0])).round(3)) # works; naive exp(1000) would overflow

### Cross-entropy: the loss behind almost every classifier and LLM

If the model is confident and right ($p_y \approx 1$), the loss is near 0. If it gives the right answer a tiny probability, the loss is large. LLM pretraining minimises exactly this, averaged over every next-token prediction in trillions of tokens.

In [ ]:
p = softmax(np.array([2.0, 1.0, 0.1]))
print(-np.log(p[0]))     # 0.417: the right class had 66% probability
print(-np.log(p[2]))     # 2.317: the right class had only 10%

### Entropy and KL divergence

It is zero only when the two are identical, and it is not symmetric. You will meet it in knowledge distillation (a small model learning to match a big model's distribution) and in RLHF, where a KL penalty stops a fine-tuned model drifting too far from where it started. A useful identity ties it together: cross-entropy equals entropy plus KL divergence, $H(P,Q) = H(P) + D_{\mathrm{KL}}(P\,\|\,Q)$.

In [ ]:
P = np.array([0.7, 0.2, 0.1])
Q = np.array([0.5, 0.3, 0.2])
H  = -(P * np.log(P)).sum()            # 0.8018
CE = -(P * np.log(Q)).sum()            # 0.8869
KL = (P * np.log(P / Q)).sum()         # 0.0851 = CE - H

### Hands-on: temperature

LLMs divide their logits by a **temperature** $T$ before softmax. For logits `[2.0, 1.0, 0.1]`, compute the distribution at $T = 0.2, 1, 5$ and its entropy. Describe in one sentence what temperature does. You will use this when you control LLM sampling in lesson 09.

#### Solution: a solution

In [ ]:
z = np.array([2.0, 1.0, 0.1])
for T in [0.2, 1, 5]:
    p = softmax(z / T)
    H = -(p * np.log(p)).sum()
    print(T, p.round(3), round(H, 3))
# 0.2 [0.993 0.007 0.   ] 0.043   nearly deterministic
# 1   [0.659 0.242 0.099] 0.832
# 5   [0.402 0.329 0.269] 1.085   close to uniform

Low temperature sharpens the distribution towards the top choice; high temperature flattens it towards uniform, so sampling becomes more random and more "creative".

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/02-math-for-ai.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.